# Background fields: the `ObjBckg` callback API

Radia applies an external (background) field via `rad.ObjBckg(callback)`, where `callback([x,y,z])` returns `[Bx,By,Bz]` in Tesla -- enabling **spatially-varying** applied fields (e.g. a quadrupole) that drive soft-iron objects. The legacy array form `ObjBckg([Bx,By,Bz])` is not supported. This notebook runs three demonstrations live.

*Notebook-coupled helpers and protected mesh/Cubit assets now live beside this notebook in `docs/background_fields/`; this is the rendered showcase.*

## See the prescribed quadrupole before adding iron

The background-field callback is easier to understand when its direction is
visible. Radia MCP's background-field workflow in `radia-analysis` owns the
operating contract. This supplement visualizes **only the prescribed source**
$\mathbf B_s=(gy,gx,0)$ with $g=10$ T/m, matching the callback below.
It does not represent the nonlinear iron response or a newly solved magnet.

The source is both divergence-free and curl-free in this current-free viewing
region, and $|\mathbf B_s|=g\sqrt{x^2+y^2}$. The arrows reveal the quadrupole
orientation; the scalar view reveals the zero on the axis. A 20 mm cube
is only a sampling domain, not a physical magnetic outer boundary or iron
body. Netgen OCC generates this viewing mesh; it is not a Cubit export.
The original material/relaxation campaign outputs below remain unchanged.


In [1]:
from pathlib import Path
import hashlib, json, math, platform, socket, sys
from datetime import datetime, timezone
from importlib.metadata import version
import ngsolve as ng
from ngsolve.webgui import Draw
ng.SetNumThreads(2)
from netgen.occ import Box, Pnt, OCCGeometry
domain=Box(Pnt(-0.01,-0.01,-0.01), Pnt(0.01,0.01,0.01))
domain.mat('source_view_not_iron')
mesh=ng.Mesh(OCCGeometry(domain).GenerateMesh(maxh=0.005))
gradient=10.0
source=ng.CF((gradient*ng.y, gradient*ng.x, 0))
magnitude=ng.Norm(source)
checks=[]
for point in [(0,0,0),(0.005,0,0),(0,0.005,0),(0.005,0.005,0)]:
    actual=list(source(mesh(*point)))
    expected=[gradient*point[1],gradient*point[0],0]
    assert all(math.isclose(x,y,abs_tol=1e-12) for x,y in zip(actual,expected))
    checks.append(dict(point_m=point,B_T=actual))
metrics=dict(reference='prescribed source only, no material response',
             samples=checks,n_elements=mesh.ne,gradient_T_per_m=gradient)
print(json.dumps(metrics,indent=2))


{
  "reference": "prescribed source only, no material response",
  "samples": [
    {
      "point_m": [
        0,
        0,
        0
      ],
      "B_T": [
        -8.673617379884035e-18,
        0.0,
        0.0
      ]
    },
    {
      "point_m": [
        0.005,
        0,
        0
      ],
      "B_T": [
        0.0,
        0.04999999999999999,
        0.0
      ]
    },
    {
      "point_m": [
        0,
        0.005,
        0
      ],
      "B_T": [
        0.05,
        0.0,
        0.0
      ]
    },
    {
      "point_m": [
        0.005,
        0.005,
        0
      ],
      "B_T": [
        0.05,
        0.05,
        0.0
      ]
    }
  ],
  "n_elements": 399,
  "gradient_T_per_m": 10.0
}


### Sampling mesh, not an iron body

In [2]:
scene = Draw(mesh, name='Quadrupole_sampling_mesh', draw_vol=True, draw_surf=True, width='100%', height='480px')
scene.GenerateHTML(filename='scene_0.html')


'\n<!DOCTYPE html>\n<html>\n    <head>\n        <title>NGSolve WebGUI</title>\n        <meta name=\'viewport\' content=\'width=device-width, user-scalable=no\'/>\n        <style>\n            body{\n                margin:0;\n                overflow:hidden;\n            }\n            canvas{\n                cursor:grab;\n                cursor:-webkit-grab;\n                cursor:-moz-grab;\n            }\n            canvas:active{\n                cursor:grabbing;\n                cursor:-webkit-grabbing;\n                cursor:-moz-grabbing;\n            }\n        </style>\n    </head>\n    <body>\n          <script src="https://cdn.jsdelivr.net/npm/webgui@0.2.39/dist/webgui.js"></script>\n          </script>\n          <script>\n            var render_data = {"gui_settings": {}, "ngsolve_version": "6.2.2606", "mesh_dim": 3, "order2d": 1, "order3d": 1, "draw_vol": null, "draw_surf": null, "objects": [], "deformation": false, "mesh_regions_2d": 6, "mesh_regions_3d": 1, "names":

### Source-field magnitude and its axial zero

In [3]:
scene = Draw(magnitude, mesh, name='Prescribed_B_magnitude_T', order=2, draw_vol=True, draw_surf=True, autoscale=False, min=0, max=0.142, clipping={'x':0,'y':0,'z':1,'dist':0}, vectors=False, width='100%', height='480px')
scene.GenerateHTML(filename='scene_1.html')


'\n<!DOCTYPE html>\n<html>\n    <head>\n        <title>NGSolve WebGUI</title>\n        <meta name=\'viewport\' content=\'width=device-width, user-scalable=no\'/>\n        <style>\n            body{\n                margin:0;\n                overflow:hidden;\n            }\n            canvas{\n                cursor:grab;\n                cursor:-webkit-grab;\n                cursor:-moz-grab;\n            }\n            canvas:active{\n                cursor:grabbing;\n                cursor:-webkit-grabbing;\n                cursor:-moz-grabbing;\n            }\n        </style>\n    </head>\n    <body>\n          <script src="https://cdn.jsdelivr.net/npm/webgui@0.2.39/dist/webgui.js"></script>\n          </script>\n          <script>\n            var render_data = {"gui_settings": {}, "ngsolve_version": "6.2.2606", "mesh_dim": 3, "order2d": 2, "order3d": 2, "draw_vol": true, "draw_surf": true, "objects": [], "deformation": false, "funcdim": 1, "show_wireframe": true, "show_mesh": t

### Source-field direction

In [4]:
scene = Draw(source, mesh, name='Prescribed_B_vector_T', order=2, draw_vol=True, draw_surf=True, autoscale=True, clipping={'x':0,'y':0,'z':1,'dist':0}, vectors={'grid_size':15}, width='100%', height='480px')
scene.GenerateHTML(filename='scene_2.html')


'\n<!DOCTYPE html>\n<html>\n    <head>\n        <title>NGSolve WebGUI</title>\n        <meta name=\'viewport\' content=\'width=device-width, user-scalable=no\'/>\n        <style>\n            body{\n                margin:0;\n                overflow:hidden;\n            }\n            canvas{\n                cursor:grab;\n                cursor:-webkit-grab;\n                cursor:-moz-grab;\n            }\n            canvas:active{\n                cursor:grabbing;\n                cursor:-webkit-grabbing;\n                cursor:-moz-grabbing;\n            }\n        </style>\n    </head>\n    <body>\n          <script src="https://cdn.jsdelivr.net/npm/webgui@0.2.39/dist/webgui.js"></script>\n          </script>\n          <script>\n            var render_data = {"gui_settings": {}, "ngsolve_version": "6.2.2606", "mesh_dim": 3, "order2d": 2, "order3d": 2, "draw_vol": true, "draw_surf": true, "objects": [], "deformation": false, "funcdim": 3, "show_wireframe": true, "show_mesh": t

In [5]:
evidence=dict(schema='radia.docs.webgui_supplement.v1',
    generated_at_utc=datetime.now(timezone.utc).isoformat(),host=socket.gethostname(),
    python_executable=sys.executable,python_version=platform.python_version(),
    versions={p:version(p) for p in ['ngsolve','netgen-mesher','anywidget','cubit-mesh-export']},
    metrics=metrics,threads=2)
Path('evidence.json').write_text(json.dumps(evidence,indent=2),encoding='utf-8')


1250

## 1. Quadrupole background + nonlinear cube

A spatially-varying **quadrupole background field** is supplied via `rad.ObjBckg(callback)` (the callback receives `[x,y,z]` and returns `[Bx,By,Bz]` in Tesla), and a nonlinear soft-iron cube (`MatSatIsoFrm`, single nested `[[H,B],...]` list) is solved in it.

In [1]:
import sys, os
sys.argv = ["notebook"]

#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Simple test to verify B->H conversion in rad.ObjBckg()

Tests that quadrupole background field defined in Tesla is correctly
converted to H field internally.
"""

import sys
import os

import numpy as np
import radia as rd

mm = 1e-3  # 1 mm in meters

print("=" * 70)
print("ObjBckg B->H Conversion Test")
print("=" * 70)

# Parameters
gradient = 10.0  # Quadrupole gradient [T/m]

print("\nParameters:")
print(f"  Quadrupole gradient: {gradient} T/m")

# ============================================================================
# Test 1: Create quadrupole background field using ObjBckg
# ============================================================================

print("\n[Test 1] Quadrupole Background Field (ObjBckg)")
print("-" * 70)

def quadrupole_field_callback(gradient):
	"""Create quadrupole field callback for rd.ObjBckg.

	Returns a callable(pos) -> [Bx, By, Bz] in Tesla.
	"""
	call_count = [0]  # Mutable to allow modification in nested function
	def field(pos):
		x, y, z = pos  # Position in meters (Radia always uses meters)
		# Quadrupole field: Bx = g*y, By = g*x, Bz = 0
		Bx = gradient * y  # [T]
		By = gradient * x  # [T]
		Bz = 0.0
		result = [Bx, By, Bz]
		# Debug: print first few calls
		call_count[0] += 1
		if call_count[0] <= 3:
			print(f"  [Callback #{call_count[0]}] pos={pos} m -> B={result} T")
		return result
	return field

quad_field = quadrupole_field_callback(gradient)
bckg_cf = rd.ObjBckg(quad_field)
print("  ObjBckg created with quadrupole field")

# ============================================================================
# Test 2: Create simple cubic element
# ============================================================================

print("\n[Test 2] Create Simple Cubic Element")
print("-" * 70)

# Small cube at center: 10mm cube centered at origin
size = 10.0 * mm
half = size / 2
# Hexahedron vertices for cube centered at [0, 0, 0] with dimensions [10, 10, 10] mm
vertices = [
	[-half, -half, -half], [half, -half, -half], [half, half, -half], [-half, half, -half],
	[-half, -half, half], [half, -half, half], [half, half, half], [-half, half, half]
]
cube = rd.ObjHexahedron(vertices, [0, 0, 0])
# Use MatSatIsoFrm for isotropic saturable material
# For soft iron-like material with high permeability
mat = rd.MatSatIsoFrm([[1596.3, 1.1488], [133.11, 0.4268], [18.713, 0.4759]])
rd.MatApl(cube, mat)
print(f"  Created {size/mm:.0f}x{size/mm:.0f}x{size/mm:.0f} mm cube with MatSatIsoFrm (nonlinear)")

# Create container with cube and background field
container = rd.ObjCnt([cube, bckg_cf])
print("  Container created with cube + ObjBckg")

# ============================================================================
# Test 3: Solve and verify field
# ============================================================================

print("\n[Test 3] Solve and Verify Field")
print("-" * 70)

print("  Solving...")
solve_result = rd.Solve(container, 1e-5, 5000)
max_abs_M = solve_result[0]  # convergence residual (max |dM|)
n_iter = int(solve_result[3])  # iteration count
print(f"  Solve result: residual={max_abs_M:.2e}, iterations={n_iter}")
if max_abs_M < 1e-5:
	print("  [OK] Solution converged")
else:
	print(f"  [WARNING] Solution may not have converged (max|dM|={max_abs_M:.2e})")

# ============================================================================
# Test 4: Compare with analytical solution
# ============================================================================

print("\n[Test 4] Compare with Analytical Solution")
print("-" * 70)

# Test points outside the cube (field ~= background + stray field from cube)
# Note: The magnetized cube produces stray fields that decay with distance.
# Points farther from the cube give closer agreement with the pure background.
test_points = [
	[50*mm, 0, 0],      # Far from cube: stray field negligible
	[0, 50*mm, 0],
	[50*mm, 50*mm, 0],
	[100*mm, 0, 0],     # Very far: essentially pure background
	[0, 100*mm, 0],
]

print("\nmu_0 = 1.25663706212e-6 T/(A/m)")
print("1/mu_0 = 795774.715459 (A/m)/T")
print()

mu_0 = 1.25663706212e-6  # T/(A/m)

print(f"{'Point (mm)':<25} {'B_Radia (T)':>30} {'H_Radia (A/m)':>30} {'B_Analytical (T)':>30} {'H=B/mu_0 (A/m)':>30} {'Error':>15}")
print("-" * 160)

for pt in test_points:
	# Get Radia fields
	B_radia = rd.Fld(container, 'b', pt)
	H_radia = rd.Fld(container, 'h', pt)

	# Analytical quadrupole field (background only, ignoring stray field from cube)
	# Positions already in meters (Radia always uses meters)
	Bx_analytical = gradient * pt[1]  # T
	By_analytical = gradient * pt[0]  # T
	Bz_analytical = 0.0

	# Analytical H field: H = B/mu_0
	Hx_analytical = Bx_analytical / mu_0
	Hy_analytical = By_analytical / mu_0
	Hz_analytical = 0.0

	# Format vectors
	B_radia_str = f"[{B_radia[0]:.6e}, {B_radia[1]:.6e}, {B_radia[2]:.6e}]"
	H_radia_str = f"[{H_radia[0]:.6e}, {H_radia[1]:.6e}, {H_radia[2]:.6e}]"
	B_analytical_str = f"[{Bx_analytical:.6e}, {By_analytical:.6e}, {Bz_analytical:.6e}]"
	H_analytical_str = f"[{Hx_analytical:.6e}, {Hy_analytical:.6e}, {Hz_analytical:.6e}]"

	# Calculate error in H field
	H_analytical = np.array([Hx_analytical, Hy_analytical, Hz_analytical])
	H_radia_arr = np.array(H_radia)
	error_H = np.linalg.norm(H_radia_arr - H_analytical)
	error_pct = error_H / (np.linalg.norm(H_analytical) + 1e-15) * 100

	pt_mm_str = str([round(p/mm) for p in pt])
	print(f"{pt_mm_str:<25} {B_radia_str:>30} {H_radia_str:>30} {B_analytical_str:>30} {H_analytical_str:>30} {error_pct:>14.4f}%")

# ============================================================================
# Test 5: Verify B/H ratio = mu_0 at multiple points
# ============================================================================

print("\n[Test 5] Verify B/H Ratio = mu_0 at Multiple Points")
print("-" * 70)

# Test at several far-field points where stray field is negligible
bh_test_points = [
	[100*mm, 0, 0],
	[0, 100*mm, 0],
	[100*mm, 100*mm, 0],
	[0, 0, 100*mm],
]

all_ok = True
for pt in bh_test_points:
	B = rd.Fld(container, 'b', pt)
	H = rd.Fld(container, 'h', pt)
	pt_mm_str = str([round(p/mm) for p in pt])

	print(f"\n  At {pt_mm_str} mm:")
	print(f"    B = [{B[0]:.8e}, {B[1]:.8e}, {B[2]:.8e}] T")
	print(f"    H = [{H[0]:.8e}, {H[1]:.8e}, {H[2]:.8e}] A/m")

	# Check B/H ratio for each non-zero component
	for comp, label in enumerate(['x', 'y', 'z']):
		if abs(H[comp]) > 1e-10:
			ratio = B[comp] / H[comp]
			rel_err = abs(ratio - mu_0) / mu_0
			status = "[OK]" if rel_err < 1e-6 else "[ERROR]"
			if rel_err >= 1e-6:
				all_ok = False
			print(f"    B_{label}/H_{label} = {ratio:.15e}  (rel_err={rel_err:.2e}) {status}")

if all_ok:
	print("\n  [OK] B/H = mu_0 within 1e-6 for all tested components")
else:
	print("\n  [ERROR] B/H != mu_0 for some components")

# ============================================================================
# Summary
# ============================================================================

print("\n" + "=" * 70)
print("Summary")
print("=" * 70)

print("\n1. ObjBckg callback returns B in Tesla")
print("2. Internal conversion: H = B / mu_0 = B x 795774.715459")
print("3. B/H = mu_0 verified at multiple far-field points and components")
print("4. Background field correctly applied via callback")

print("\n" + "=" * 70)
print("Test Complete")
print("=" * 70)

rd.UtiDelAll()


ObjBckg B->H Conversion Test

Parameters:
  Quadrupole gradient: 10.0 T/m

[Test 1] Quadrupole Background Field (ObjBckg)
----------------------------------------------------------------------
  ObjBckg created with quadrupole field

[Test 2] Create Simple Cubic Element
----------------------------------------------------------------------
  Created 10x10x10 mm cube with MatSatIsoFrm (nonlinear)
  Container created with cube + ObjBckg

[Test 3] Solve and Verify Field
----------------------------------------------------------------------
  Solving...
  [Callback #1] pos=[3.1830988618086527e-10, 3.1830988618086527e-10, 3.183098863977057e-10] m -> B=[3.183098861808653e-09, 3.183098861808653e-09, 0.0] T
  Solve result: residual=6.75e-09, iterations=1
  [OK] Solution converged

[Test 4] Compare with Analytical Solution
----------------------------------------------------------------------

mu_0 = 1.25663706212e-6 T/(A/m)
1/mu_0 = 795774.715459 (A/m)/T

Point (mm)                            

## 2. Sphere in a quadrupole background

Same `ObjBckg` quadrupole driving a soft-iron sphere -- the magnetisation follows the spatially-varying applied field.

In [2]:
import sys, os
sys.argv = ["notebook"]

#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Analytical Solution Comparison for Quadrupole Field with ObjBckg

Tests magnetizable cube (approximating sphere) in quadrupole background field.
Compares Radia numerical solution with analytical quadrupole field
at points outside the cube.
"""

import sys
import os

import numpy as np
import radia as rd

mm = 1e-3  # 1 mm in meters

print("=" * 80)
print("Quadrupole Field - Analytical Solution Comparison")
print("=" * 80)

# Parameters
gradient = 10.0  # T/m
R_sphere = 5.0 * mm   # half-size of cube (sphere approximation)

print("\nParameters:")
print(f"  Cube half-size: {R_sphere/mm:.1f} mm")
print(f"  Quadrupole gradient: {gradient} T/m")

# ============================================================================
# Create Geometry
# ============================================================================

print("\n[Step 1] Creating Geometry")
print("-" * 80)

# Simple cubic approximation of sphere: 10mm cube centered at origin
size = 2 * R_sphere  # 10mm cube
half = size / 2
# Hexahedron vertices for cube centered at [0, 0, 0] with dimensions [10, 10, 10] mm
vertices = [
	[-half, -half, -half], [half, -half, -half], [half, half, -half], [-half, half, -half],
	[-half, -half, half], [half, -half, half], [half, half, half], [-half, half, half]
]
cube = rd.ObjHexahedron(vertices, [0, 0, 0])
mat = rd.MatSatIsoFrm([[1596.3, 1.1488], [133.11, 0.4268], [18.713, 0.4759]])
rd.MatApl(cube, mat)
print(f"  Created {size/mm:.0f}x{size/mm:.0f}x{size/mm:.0f} mm cube with MatSatIsoFrm (nonlinear)")

# ============================================================================
# Create Quadrupole Background Field
# ============================================================================

print("\n[Step 2] Creating Quadrupole Background Field")
print("-" * 80)

def quadrupole_field(pos):
	"""Quadrupole field: Bx = g*y, By = g*x, Bz = 0"""
	x, y, z = pos  # Position in meters (Radia always uses meters)
	Bx = gradient * y  # [T]
	By = gradient * x  # [T]
	Bz = 0.0
	return [Bx, By, Bz]

bckg_cf = rd.ObjBckg(quadrupole_field)
print("  Quadrupole field created: Bx = g*y, By = g*x")

# Container with cube and background field
container = rd.ObjCnt([cube, bckg_cf])
print("  Container created")

# ============================================================================
# Solve
# ============================================================================

print("\n[Step 3] Solving Magnetostatic Problem")
print("-" * 80)

print("  Solving...")
solve_result = rd.Solve(container, 1e-5, 5000)
max_abs_M = solve_result[0]  # convergence residual (max |dM|)
n_iter = int(solve_result[3])  # iteration count
print(f"  Solve result: residual={max_abs_M:.2e}, iterations={n_iter}")
if max_abs_M < 1e-5:
	print("  [OK] Solution converged")
else:
	print(f"  [WARNING] Solution may not have converged (max|dM|={max_abs_M:.2e})")

# ============================================================================
# Compare with Analytical Solution
# ============================================================================

print("\n[Step 4] Compare with Analytical Quadrupole Field")
print("-" * 80)

# Test points outside the cube (field ~= background + stray field from cube)
# Note: The magnetized cube produces stray fields that decay with distance.
# Points farther from the cube give closer agreement with the pure background.
test_points = [
	# Along X-axis (y=0, z=0)
	[20*mm, 0, 0],    # r = 20mm
	[30*mm, 0, 0],    # r = 30mm
	[50*mm, 0, 0],    # r = 50mm
	[100*mm, 0, 0],   # r = 100mm
	# Along Y-axis (x=0, z=0)
	[0, 20*mm, 0],    # r = 20mm
	[0, 30*mm, 0],    # r = 30mm
	[0, 50*mm, 0],    # r = 50mm
	[0, 100*mm, 0],   # r = 100mm
	# Diagonal points
	[20*mm, 20*mm, 0],   # r = 28.28mm
	[50*mm, 50*mm, 0],   # r = 70.71mm
	[100*mm, 100*mm, 0], # r = 141.42mm
]

print("\nAnalytical quadrupole field: Bx = g*y, By = g*x, Bz = 0")
print(f"where g = {gradient} T/m")
print(f"\nComparison at points outside cube (r > {R_sphere/mm:.1f} mm):")
print()
print(f"{'Point (mm)':<25} {'r (mm)':>8} | {'B_Radia (T)':^35} | {'B_Analytical (T)':^35} | {'|ΔB| (T)':>12} {'Error (%)':>10}")
print("-" * 140)

errors = []
for pt in test_points:
	# Calculate distance from center
	r = np.sqrt(pt[0]**2 + pt[1]**2 + pt[2]**2)

	# Radia solution
	B_radia = rd.Fld(container, 'b', pt)

	# Analytical quadrupole field (positions already in meters)
	B_analytical = np.array([gradient * pt[1], gradient * pt[0], 0.0])

	# Calculate error
	B_radia_arr = np.array(B_radia)
	delta_B = B_radia_arr - B_analytical
	error_mag = np.linalg.norm(delta_B)
	B_analytical_mag = np.linalg.norm(B_analytical)

	if B_analytical_mag > 1e-10:
		error_pct = error_mag / B_analytical_mag * 100
	else:
		error_pct = 0.0

	errors.append(error_pct)

	# Format output
	pt_mm = [p/mm for p in pt]
	B_radia_str = f"[{B_radia[0]:8.5f}, {B_radia[1]:8.5f}, {B_radia[2]:8.5f}]"
	B_analytical_str = f"[{B_analytical[0]:8.5f}, {B_analytical[1]:8.5f}, {B_analytical[2]:8.5f}]"

	print(f"{str([f'{p:.0f}' for p in pt_mm]):<25} {r/mm:8.2f} | {B_radia_str:^35} | {B_analytical_str:^35} | {error_mag:12.6e} {error_pct:9.4f}%")

# ============================================================================
# Statistics
# ============================================================================

print("\n[Step 5] Error Statistics")
print("-" * 80)

errors_arr = np.array(errors)
print("\nError statistics:")
print(f"  Mean error:    {errors_arr.mean():.4f}%")
print(f"  Median error:  {np.median(errors_arr):.4f}%")
print(f"  Max error:     {errors_arr.max():.4f}%")
print(f"  Min error:     {errors_arr.min():.4f}%")
print(f"  Std deviation: {errors_arr.std():.4f}%")

# Group by distance
print("\nError vs. distance from center:")
distances_mm = [20, 30, 50, 100]
for d_mm in distances_mm:
	d = d_mm * mm
	# Find errors for points at this distance (+-1mm tolerance)
	d_errors = []
	for i, pt in enumerate(test_points):
		r = np.sqrt(pt[0]**2 + pt[1]**2 + pt[2]**2)
		if abs(r - d) < 1.5*mm:  # Tolerance for diagonal points
			d_errors.append(errors[i])

	if d_errors:
		avg_error = np.mean(d_errors)
		print(f"  r ~ {d_mm:2d} mm: {avg_error:6.4f}% average error ({len(d_errors)} points)")

# ============================================================================
# Physical Interpretation
# ============================================================================

print("\n[Step 6] Physical Interpretation")
print("-" * 80)

print(f"\nExpected behavior:")
print(f"  1. Far from cube (r >> {R_sphere/mm:.1f} mm): B_Radia ~ B_Analytical (pure quadrupole)")
print(f"  2. Near cube (r ~ {R_sphere/mm:.1f} mm): Small distortion due to magnetizable material")
print("  3. Error should decrease as 1/r^2 (dipole perturbation)")

# Compute far-field errors (r >= 50mm)
far_field_errors = []
for i, pt in enumerate(test_points):
	r = np.sqrt(pt[0]**2 + pt[1]**2 + pt[2]**2)
	if r >= 50*mm:
		far_field_errors.append(errors[i])
far_field_mean = np.mean(far_field_errors) if far_field_errors else float('inf')

if far_field_mean < 1.0:
	print(f"\n  [OK] Far-field accuracy (r>=50mm): {far_field_mean:.4f}% < 1%")
else:
	print(f"\n  [WARNING] Far-field error higher than expected: {far_field_mean:.4f}%")

if errors_arr[-1] < errors_arr[0]:
	print("  [OK] Error decreases with distance (as expected)")
else:
	print("  [WARNING] Error does not decrease with distance")

# ============================================================================
# Summary
# ============================================================================

print("\n" + "=" * 80)
print("Summary")
print("=" * 80)

print("\n1. ObjBckg successfully implements quadrupole background field")
print(f"2. Radia numerical solution compared with analytical quadrupole at {len(test_points)} points")
print(f"3. Average error: {errors_arr.mean():.4f}%")
print(f"4. Far-field agreement (r>=50mm): {far_field_mean:.4f}%")

if errors_arr.mean() < 5.0:
	print("\n[OK] Good agreement with analytical solution (avg error < 5%)")
elif errors_arr.mean() < 15.0:
	print("\n[OK] Reasonable agreement with analytical solution (avg error < 15%)")
else:
	print("\n[WARNING] Significant deviation from analytical solution")

print("\n" + "=" * 80)
print("Test Complete")
print("=" * 80)

rd.UtiDelAll()


Quadrupole Field - Analytical Solution Comparison

Parameters:
  Cube half-size: 5.0 mm
  Quadrupole gradient: 10.0 T/m

[Step 1] Creating Geometry
--------------------------------------------------------------------------------
  Created 10x10x10 mm cube with MatSatIsoFrm (nonlinear)

[Step 2] Creating Quadrupole Background Field
--------------------------------------------------------------------------------
  Quadrupole field created: Bx = g*y, By = g*x
  Container created

[Step 3] Solving Magnetostatic Problem
--------------------------------------------------------------------------------
  Solving...
  Solve result: residual=6.75e-09, iterations=1
  [OK] Solution converged

[Step 4] Compare with Analytical Quadrupole Field
--------------------------------------------------------------------------------

Analytical quadrupole field: Bx = g*y, By = g*x, Bz = 0
where g = 10.0 T/m

Comparison at points outside cube (r > 5.0 mm):

Point (mm)                  r (mm) |             B_Ra

## 3. Permeability sweep

Solved magnetisation / field vs relative permeability under the background field.

In [3]:
import sys, os
sys.argv = ["notebook"]

#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Permeability Comparison - Analytical Solution Test

Compares Radia numerical solutions with analytical quadrupole field
for different permeability values (mu_r).

Tests magnetizable cube (approximating sphere) in quadrupole background field with:
- mu_r = 10 (low permeability)
- mu_r = 100 (medium permeability)
- mu_r = 1000 (high permeability - soft iron)
"""

import sys
import os

import numpy as np
import radia as rd

mm = 1e-3  # 1 mm in meters

print("=" * 80)
print("Permeability Comparison - Analytical Solution Test")
print("=" * 80)

# Test parameters
gradient = 10.0  # T/m
R_sphere = 5.0 * mm   # half-size of cube (sphere approximation)
permeability_values = [10, 100, 1000]

# Test points outside the cube (field ~= background + stray field from cube)
# Note: The magnetized cube produces stray fields that decay with distance.
# Points farther from the cube give closer agreement with the pure background.
test_points = [
	# Along X-axis (y=0, z=0)
	[20*mm, 0, 0],    # r = 20mm
	[30*mm, 0, 0],    # r = 30mm
	[50*mm, 0, 0],    # r = 50mm
	[100*mm, 0, 0],   # r = 100mm
	# Along Y-axis (x=0, z=0)
	[0, 20*mm, 0],    # r = 20mm
	[0, 30*mm, 0],    # r = 30mm
	[0, 50*mm, 0],    # r = 50mm
	[0, 100*mm, 0],   # r = 100mm
	# Diagonal points
	[20*mm, 20*mm, 0],   # r = 28.28mm
	[50*mm, 50*mm, 0],   # r = 70.71mm
	[100*mm, 100*mm, 0], # r = 141.42mm
]

def quadrupole_field(pos):
	"""Quadrupole field: Bx = g*y, By = g*x, Bz = 0"""
	x, y, z = pos  # Position in meters (Radia always uses meters)
	Bx = gradient * y  # [T]
	By = gradient * x  # [T]
	Bz = 0.0
	return [Bx, By, Bz]

# Store results for all permeability values
all_results = {}

# ============================================================================
# Run Tests for Each Permeability Value
# ============================================================================

for mu_r in permeability_values:
	print(f"\n{'=' * 80}")
	print(f"Testing with mu_r = {mu_r}")
	print(f"{'=' * 80}")

	print("\nParameters:")
	print(f"  Cube half-size: {R_sphere/mm:.1f} mm")
	print(f"  Relative permeability: {mu_r}")
	print(f"  Quadrupole gradient: {gradient} T/m")

	# Create Geometry
	print("\n[Step 1] Creating Geometry")
	print("-" * 80)

	rd.UtiDelAll()  # Clear all previous objects

	# Simple cubic approximation of sphere: 10mm cube centered at origin
	size = 2 * R_sphere  # 10mm cube
	half = size / 2
	# Hexahedron vertices for cube centered at [0, 0, 0] with dimensions [10, 10, 10] mm
	vertices = [
		[-half, -half, -half], [half, -half, -half], [half, half, -half], [-half, half, -half],
		[-half, -half, half], [half, -half, half], [half, half, half], [-half, half, half]
	]
	cube = rd.ObjHexahedron(vertices, [0, 0, 0])

	# Use linear material with specified permeability
	mat = rd.MatLin(mu_r)
	rd.MatApl(cube, mat)
	print(f"  Created {size/mm:.0f}x{size/mm:.0f}x{size/mm:.0f} mm cube with MatLin(mu_r={mu_r})")

	# Create Quadrupole Background Field
	print("\n[Step 2] Creating Quadrupole Background Field")
	print("-" * 80)

	bckg_cf = rd.ObjBckg(quadrupole_field)
	print("  Quadrupole field created: Bx = g*y, By = g*x")

	# Container with cube and background field
	container = rd.ObjCnt([cube, bckg_cf])
	print("  Container created")

	# Solve
	print("\n[Step 3] Solving Magnetostatic Problem")
	print("-" * 80)

	print("  Solving...")
	solve_result = rd.Solve(container, 1e-5, 5000)
	max_abs_M = solve_result[0]  # convergence residual (max |dM|)
	n_iter = int(solve_result[3])  # iteration count
	print(f"  Solve result: residual={max_abs_M:.2e}, iterations={n_iter}")
	if max_abs_M < 1e-5:
		print("  [OK] Solution converged")
	else:
		print(f"  [WARNING] Solution may not have converged (max|dM|={max_abs_M:.2e})")

	# Compare with Analytical Solution
	print("\n[Step 4] Compare with Analytical Quadrupole Field")
	print("-" * 80)

	print(f"\nComparison at points outside cube (r > {R_sphere/mm:.1f} mm):")
	print()
	print(f"{'Point (mm)':<25} {'r (mm)':>8} | {'B_Radia (T)':^35} | {'B_Analytical (T)':^35} | {'|Delta B| (T)':>12} {'Error (%)':>10}")
	print("-" * 140)

	errors = []
	for pt in test_points:
		# Calculate distance from center
		r = np.sqrt(pt[0]**2 + pt[1]**2 + pt[2]**2)

		# Radia solution
		B_radia = rd.Fld(container, 'b', pt)

		# Analytical quadrupole field (positions already in meters)
		B_analytical = np.array([gradient * pt[1], gradient * pt[0], 0.0])

		# Calculate error
		B_radia_arr = np.array(B_radia)
		delta_B = B_radia_arr - B_analytical
		error_mag = np.linalg.norm(delta_B)
		B_analytical_mag = np.linalg.norm(B_analytical)

		if B_analytical_mag > 1e-10:
			error_pct = error_mag / B_analytical_mag * 100
		else:
			error_pct = 0.0

		errors.append(error_pct)

		# Format output
		pt_mm = [p/mm for p in pt]
		B_radia_str = f"[{B_radia[0]:8.5f}, {B_radia[1]:8.5f}, {B_radia[2]:8.5f}]"
		B_analytical_str = f"[{B_analytical[0]:8.5f}, {B_analytical[1]:8.5f}, {B_analytical[2]:8.5f}]"

		print(f"{str([f'{p:.0f}' for p in pt_mm]):<25} {r/mm:8.2f} | {B_radia_str:^35} | {B_analytical_str:^35} | {error_mag:12.6e} {error_pct:9.4f}%")

	# Statistics
	print(f"\n[Step 5] Error Statistics for mu_r = {mu_r}")
	print("-" * 80)

	errors_arr = np.array(errors)
	print("\nError statistics:")
	print(f"  Mean error:    {errors_arr.mean():.4f}%")
	print(f"  Median error:  {np.median(errors_arr):.4f}%")
	print(f"  Max error:     {errors_arr.max():.4f}%")
	print(f"  Min error:     {errors_arr.min():.4f}%")
	print(f"  Std deviation: {errors_arr.std():.4f}%")

	# Store results
	all_results[mu_r] = {
		'errors': errors_arr,
		'mean': errors_arr.mean(),
		'median': np.median(errors_arr),
		'max': errors_arr.max(),
		'min': errors_arr.min(),
		'std': errors_arr.std(),
	}

# ============================================================================
# Summary Comparison Table
# ============================================================================

print(f"\n{'=' * 80}")
print("Summary: Permeability Comparison")
print(f"{'=' * 80}")

print("\nError statistics for different permeability values:")
print()
print(f"{'mu_r':>6} | {'Mean (%)':>10} {'Median (%)':>12} {'Max (%)':>10} {'Min (%)':>10} {'Std (%)':>10}")
print("-" * 80)

for mu_r in permeability_values:
	res = all_results[mu_r]
	print(f"{mu_r:6d} | {res['mean']:10.4f} {res['median']:12.4f} {res['max']:10.4f} {res['min']:10.4f} {res['std']:10.4f}")

# ============================================================================
# Physical Interpretation
# ============================================================================

print(f"\n{'=' * 80}")
print("Physical Interpretation")
print(f"{'=' * 80}")

print("\nKey Observations:")
print()
print(f"1. Near-field distortion (r ~ {R_sphere*2/mm:.0f} mm):")
for mu_r in permeability_values:
	res = all_results[mu_r]
	# First point along X-axis: r=20mm (index 0)
	error_near = res['errors'][0]
	print(f"   mu_r = {mu_r:4d}: {error_near:6.2f}% error at r=20mm")

print("\n2. Far-field accuracy (r >= 50 mm):")
for mu_r in permeability_values:
	res = all_results[mu_r]
	# Select points at r >= 50mm by checking actual distances
	far_errors = []
	for i, pt in enumerate(test_points):
		r = np.sqrt(pt[0]**2 + pt[1]**2 + pt[2]**2)
		if r >= 50*mm:
			far_errors.append(res['errors'][i])
	avg_far_field = np.mean(far_errors) if far_errors else float('inf')
	print(f"   mu_r = {mu_r:4d}: {avg_far_field:6.4f}% average error")

print("\n3. Overall accuracy:")
for mu_r in permeability_values:
	res = all_results[mu_r]
	if res['mean'] < 1.0:
		status = "[EXCELLENT]"
	elif res['mean'] < 5.0:
		status = "[GOOD]     "
	else:
		status = "[MODERATE] "
	print(f"   mu_r = {mu_r:4d}: {status} {res['mean']:6.4f}% average error")

print("\n4. Permeability effect:")
print("   Higher permeability -> Stronger field distortion near cube")
print("   But far-field accuracy remains excellent for all mu_r values")
print("   Error scaling follows 1/r^2 behavior (dipole perturbation)")

# ============================================================================
# Final Summary
# ============================================================================

print(f"\n{'=' * 80}")
print("Final Summary")
print(f"{'=' * 80}")

print("\n1. ObjBckg successfully implements quadrupole background field")
print(f"2. Tested with {len(permeability_values)} different permeability values: {permeability_values}")
print("3. All tests show good agreement with analytical solution")
print("4. Near-field distortion increases with permeability (as expected)")

best_mu = min(all_results.keys(), key=lambda k: all_results[k]['mean'])
worst_mu = max(all_results.keys(), key=lambda k: all_results[k]['mean'])

print(f"\nBest overall accuracy: mu_r = {best_mu} ({all_results[best_mu]['mean']:.4f}% average error)")
print(f"Largest distortion: mu_r = {worst_mu} ({all_results[worst_mu]['mean']:.4f}% average error)")
print(f"Difference: {all_results[worst_mu]['mean'] - all_results[best_mu]['mean']:.4f}%")

print(f"\n{'=' * 80}")
print("Test Complete")
print(f"{'=' * 80}")

rd.UtiDelAll()


Permeability Comparison - Analytical Solution Test

Testing with mu_r = 10

Parameters:
  Cube half-size: 5.0 mm
  Relative permeability: 10
  Quadrupole gradient: 10.0 T/m

[Step 1] Creating Geometry
--------------------------------------------------------------------------------
  Created 10x10x10 mm cube with MatLin(mu_r=10)

[Step 2] Creating Quadrupole Background Field
--------------------------------------------------------------------------------
  Quadrupole field created: Bx = g*y, By = g*x
  Container created

[Step 3] Solving Magnetostatic Problem
--------------------------------------------------------------------------------
  Solving...
  Solve result: residual=1.13e-08, iterations=1
  [OK] Solution converged

[Step 4] Compare with Analytical Quadrupole Field
--------------------------------------------------------------------------------

Comparison at points outside cube (r > 5.0 mm):

Point (mm)                  r (mm) |             B_Radia (T)             |          B

## Durable JSON Record

The notebook writes a compact JSON record with runtime versions, script source hashes, asset hashes, and successful execution summaries for the three non-Cubit helpers. The Cubit mesh-generation helper is archived and hashed but not executed here.

In [4]:

from pathlib import Path
import datetime as _dt
import hashlib
import importlib.metadata as _metadata
import json
import platform
import subprocess
import sys

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'quadrupole_analytical.py').exists():
    NOTEBOOK_DIR = Path('docs/background_fields').resolve()


def _sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def _version(name):
    try:
        return _metadata.version(name)
    except Exception:
        return None

scripts_to_run = [
    'quadrupole_analytical.py',
    'sphere_in_quadrupole.py',
    'permeability_comparison.py',
]
script_runs = []
for script in scripts_to_run:
    proc = subprocess.run(
        [sys.executable, str(NOTEBOOK_DIR / script)],
        cwd=NOTEBOOK_DIR,
        text=True,
        capture_output=True,
        timeout=120,
    )
    tail = proc.stdout.splitlines()[-14:]
    script_runs.append({
        'script': script,
        'returncode': proc.returncode,
        'stdout_sha256': hashlib.sha256(proc.stdout.encode('utf-8')).hexdigest(),
        'stdout_tail': tail,
    })
    assert proc.returncode == 0, proc.stderr[-2000:]

source_files = [
    'cubit_to_nastran.py',
    'quadrupole_analytical.py',
    'sphere_in_quadrupole.py',
    'permeability_comparison.py',
    'README.md',
]
assets = ['sphere.bdf', 'sphere_nastran_field_mu.pvsm']
result = {
    'schema': 'radia.docs.background_fields.results.v1',
    'generated_at_utc': _dt.datetime.now(_dt.timezone.utc).isoformat(timespec='seconds').replace('+00:00', 'Z'),
    'versions': {
        'python_version': sys.version.split()[0],
        'python_executable': sys.executable,
        'platform': platform.platform(),
        'radia_version': _version('radia'),
        'numpy_version': _version('numpy'),
    },
    'topic_dir': 'docs/background_fields',
    'source_hashes': {
        name: _sha256(NOTEBOOK_DIR / name)
        for name in source_files
        if (NOTEBOOK_DIR / name).exists()
    },
    'asset_hashes': {
        name: {'sha256': _sha256(NOTEBOOK_DIR / name), 'bytes': (NOTEBOOK_DIR / name).stat().st_size}
        for name in assets
        if (NOTEBOOK_DIR / name).exists()
    },
    'script_runs': script_runs,
    'cubit_helper': {
        'script': 'cubit_to_nastran.py',
        'executed': False,
        'reason': 'requires Coreform Cubit; archived and hashed here, executable outside notebook when Cubit is available',
    },
    'checks': {
        'all_non_cubit_scripts_passed': all(row['returncode'] == 0 for row in script_runs),
        'mesh_assets_present': all((NOTEBOOK_DIR / name).exists() for name in assets),
    },
}
assert result['checks']['all_non_cubit_scripts_passed']
assert result['checks']['mesh_assets_present']
out = NOTEBOOK_DIR / 'background_fields_results.json'
out.write_text(json.dumps(result, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print(f"wrote {out.relative_to(NOTEBOOK_DIR)}")
print(json.dumps(result['checks'], sort_keys=True))


wrote background_fields_results.json
{"all_non_cubit_scripts_passed": true, "mesh_assets_present": true}


## Method, implementation and scope

The callback supplies a prescribed source field, not a solution for the iron.
For the quadrupole used below,

$$\mathbf B_s=(g y,g x,0),\qquad \nabla\cdot\mathbf B_s=0,
\quad\nabla\times\mathbf B_s=0,\qquad \mathbf H_s=\mathbf B_s/\mu_0.$$

The last conversion matters: the callback returns tesla, while the material
solve uses field strength. Magnetostatics requires
$\mathbf B=\mu_0(\mathbf H+\mathbf M)$; the demagnetizing field couples the
unknown magnetization back to the applied field. The integral-field ancestry
is [@chubar1998three]; the Maxwell identities are [@jackson1998classical].
The spatial callback and its unit conversion are Radia implementation choices,
not a new quadrupole law.

`ObjBckg`, `MatSatIsoFrm` and the cube/sphere solves below connect that model to
code. Read the printed B-to-H checks and permeability sweep separately from
accuracy of the nonlinear iron solution: callback acceptance alone does not
prove mesh or nonlinear convergence. The prescribed source neglects changes
to the source caused by the iron. Ask the Radia MCP field/material capability
for current execution instructions; Python here is the LLM-driven reproduction.


## References

Generated from the canonical bibliography: [background_fields.bbl](background_fields.bbl).

<h3 class='likesectionHead' id='references'><a id='x1-1000'></a>References</h3>
<!-- l. 2 --><p class='noindent'>
   </p><section class='thebibliography' role='doc-bibliography'><dl><dt>
 [1]</dt><dd><a id='Xchubar1998three'></a>O. Chubar,               P. Elleaume,               and               J. Chavanne,
   “A three-dimensional magnetostatics computer code for insertion devices,”
   <span class='ecti-1000'>Journal of synchrotron radiation</span>, vol. 5, no. 3, pp. 481–484, 1998.
   </dd><dt>
 [2]</dt><dd><a id='Xjackson1998classical'></a>J. D. Jackson, <span class='ecti-1000'>Classical Electrodynamics</span>, 3rd ed.   Wiley, 1998.
   </dd></dl></section>
